In [1]:
import pandas as pd
import vivarium_inputs
import gbd_mapping

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "india"
vehicle = "rice"
scenario = "intervention"

In [3]:
# Parameters
location = "india"
vehicle = "rice"
scenario = "intervention"


In [4]:
def aggregate_by_scenario(df):
    return df.groupby(["scenario", "input_draw", "wealth_quintile","sub_entity"]).value.sum().groupby(["scenario", "wealth_quintile","sub_entity"]).mean()

In [5]:
preg_anemia_prev = pd.read_parquet(f"0100_rescale_results/pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet")
preg_anemia_prev

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,lowest,baseline,2,0,0.000000
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,second,baseline,2,0,0.000000
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,middle,baseline,2,0,0.000000
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,fourth,baseline,2,0,112.999269
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,highest,baseline,2,0,93.670447
...,...,...,...,...,...,...,...,...,...,...,...
35995,person_time,impairment,anemia,severe,95_plus,severe,lowest,baseline,3,0,0.000000
35996,person_time,impairment,anemia,severe,95_plus,severe,second,baseline,3,0,0.000000
35997,person_time,impairment,anemia,severe,95_plus,severe,middle,baseline,3,0,0.000000
35998,person_time,impairment,anemia,severe,95_plus,severe,fourth,baseline,3,0,0.000000


In [6]:
preg_anemia_prev.sub_entity.value_counts()
# TODO: Double-check that sub_entity is the best variable to use for anemia status? (Or is anemia_status_by_birth better?) 

mild          9000
moderate      9000
not_anemic    9000
severe        9000
Name: sub_entity, dtype: int64

In [7]:
preg_anemia_prev.anemia_status_at_birth.value_counts()

invalid       7200
mild          7200
moderate      7200
not_anemic    7200
severe        7200
Name: anemia_status_at_birth, dtype: int64

In [8]:
preg_anemia_prev.groupby("scenario").random_seed.nunique()

scenario
baseline        10
intervention    10
Name: random_seed, dtype: int64

In [9]:
preg_anemia_prev_by_scenario = aggregate_by_scenario(preg_anemia_prev)
preg_anemia_prev_by_scenario

scenario      wealth_quintile  sub_entity
baseline      fourth           mild          4.428322e+05
                               moderate      4.888274e+05
                               not_anemic    1.063625e+06
                               severe        3.355038e+04
              highest          mild          4.061283e+05
                               moderate      3.562882e+05
                               not_anemic    1.194640e+06
                               severe        2.088108e+04
              lowest           mild          6.741982e+05
                               moderate      8.980201e+05
                               not_anemic    1.226359e+06
                               severe        7.483525e+04
              middle           mild          5.028735e+05
                               moderate      5.898235e+05
                               not_anemic    1.065565e+06
                               severe        4.268845e+04
              second          

In [10]:
preg_total_person_time = preg_anemia_prev_by_scenario.groupby(level=['scenario', 'wealth_quintile']).sum()
preg_total_person_time

scenario      wealth_quintile
baseline      fourth             2.028835e+06
              highest            1.977938e+06
              lowest             2.873412e+06
              middle             2.200951e+06
              second             2.441965e+06
intervention  fourth             2.028835e+06
              highest            1.977938e+06
              lowest             2.873412e+06
              middle             2.200951e+06
              second             2.441965e+06
Name: value, dtype: float64

In [11]:
preg_anemia_prev_by_scenario_and_state = preg_anemia_prev_by_scenario[preg_anemia_prev_by_scenario.index.get_level_values('sub_entity') != 'not_anemic']
preg_anemia_prev_by_scenario_and_state = preg_anemia_prev_by_scenario_and_state / preg_total_person_time
preg_anemia_prev_by_scenario_and_state
# Sum the person-times of each anemia state together (total person-time), then divide the person-time within each state (sub_entity) by the total person-time across all states
# to get prevalence rate of each anemia state

scenario      wealth_quintile  sub_entity
baseline      fourth           mild          0.218269
                               moderate      0.240940
                               severe        0.016537
              highest          mild          0.205329
                               moderate      0.180131
                               severe        0.010557
              lowest           mild          0.234633
                               moderate      0.312527
                               severe        0.026044
              middle           mild          0.228480
                               moderate      0.267986
                               severe        0.019395
              second           mild          0.229250
                               moderate      0.297473
                               severe        0.022973
intervention  fourth           mild          0.214763
                               moderate      0.234388
                               severe   

In [12]:
# Now that I've calculated the prevalence of each anemia state seaparately, I can sum them together and stratify by just scenario and wealth quintile. 
preg_anemia_prev_by_scenario = preg_anemia_prev_by_scenario_and_state.groupby(level=['scenario', 'wealth_quintile']).sum() 
preg_anemia_prev_by_scenario

scenario      wealth_quintile
baseline      fourth             0.475746
              highest            0.396017
              lowest             0.573205
              middle             0.515861
              second             0.549696
intervention  fourth             0.465354
              highest            0.385076
              lowest             0.564658
              middle             0.507241
              second             0.541283
Name: value, dtype: float64

In [13]:
preg_anemia_prev_averted = preg_anemia_prev_by_scenario.loc["baseline"] - preg_anemia_prev_by_scenario.loc[scenario]
preg_anemia_prev_averted 
# This should be in RATE-space. 

wealth_quintile
fourth     0.010392
highest    0.010941
lowest     0.008547
middle     0.008620
second     0.008413
Name: value, dtype: float64

In [14]:
preg_pct_anemia_prev_averted = (preg_anemia_prev_averted / preg_anemia_prev_by_scenario.loc["baseline"]) * 100 
preg_pct_anemia_prev_averted

wealth_quintile
fourth     2.184321
highest    2.762788
lowest     1.491027
middle     1.670973
second     1.530432
Name: value, dtype: float64

In [15]:
df_preg_anemia = pd.DataFrame(preg_anemia_prev_averted).rename(columns={'value':'anemia_prev_averted'})
df_preg_anemia

,anemia_prev_averted
wealth_quintile,
fourth,0.010392
highest,0.010941
lowest,0.008547
middle,0.008620
second,0.008413


In [16]:
df_preg_anemia['pct_anemia_prev_averted'] = preg_pct_anemia_prev_averted
df_preg_anemia

,anemia_prev_averted,pct_anemia_prev_averted
wealth_quintile,,
fourth,0.010392,2.184321
highest,0.010941,2.762788
lowest,0.008547,1.491027
middle,0.008620,1.670973
second,0.008413,1.530432


In [17]:
df_preg_anemia = df_preg_anemia.assign(pregnant="pregnant").set_index("pregnant", append=True)

In [18]:
non_preg_anemia_prev = pd.read_parquet(f"../0400_non_pregnant_anemia_model/{vehicle}/{location}/{scenario}/anemia_prevalence.parquet")
non_preg_anemia_prev = non_preg_anemia_prev.set_index([c for c in non_preg_anemia_prev.columns if c != "value"]).assign(pregnant="not_pregnant").set_index("pregnant", append=True).value
non_preg_anemia_prev

age_start  age_end     sex     wealth_quintile  scenario      pregnant    
0.0        0.019178    Female  fourth           baseline      not_pregnant    0.917220
                               highest          baseline      not_pregnant    0.900539
                               lowest           baseline      not_pregnant    0.965487
                               middle           baseline      not_pregnant    0.940266
                               second           baseline      not_pregnant    0.947353
                                                                                ...   
95.0       125.000000  Male    fourth           intervention  not_pregnant    0.757056
                               highest          intervention  not_pregnant    0.756073
                               lowest           intervention  not_pregnant    0.781134
                               middle           intervention  not_pregnant    0.760664
                               second           interve

In [19]:
pop = pd.read_csv(f'../0100_data_prep/results/population/stratified/{location}.csv')
pop = pop.set_index([c for c in pop.columns if c != "value"]).value
pop

sex     age_start  age_end     pregnant      wealth_quintile
Female  0.0        0.019178    not_pregnant  lowest             49649.700497
                                             second             43575.622144
                                             middle             38689.356750
                                             fourth             36440.833935
                                             highest            29202.585338
                                                                    ...     
Male    95.0       125.000000  not_pregnant  lowest             15176.692284
                                             second             15848.509818
                                             middle             16370.176208
                                             fourth             17265.313670
                                             highest            21289.473456
Name: value, Length: 285, dtype: float64

In [20]:
non_preg_anemia_prev_by_scenario = (non_preg_anemia_prev * pop).groupby(["scenario", "wealth_quintile"]).sum() / pop.groupby(["wealth_quintile"]).sum()
non_preg_anemia_prev_by_scenario

scenario      wealth_quintile
baseline      fourth             0.386766
              highest            0.361788
              lowest             0.456091
              middle             0.410115
              second             0.420173
intervention  fourth             0.374551
              highest            0.345479
              lowest             0.445933
              middle             0.398886
              second             0.409973
Name: value, dtype: float64

In [21]:
non_preg_anemia_prev_averted = non_preg_anemia_prev_by_scenario.loc["baseline"] - non_preg_anemia_prev_by_scenario.loc["intervention"]
non_preg_anemia_prev_averted

wealth_quintile
fourth     0.012215
highest    0.016309
lowest     0.010157
middle     0.011229
second     0.010200
Name: value, dtype: float64

In [22]:
non_preg_pct_anemia_prev_averted = (non_preg_anemia_prev_averted / non_preg_anemia_prev_by_scenario.loc["baseline"]) * 100 
non_preg_pct_anemia_prev_averted

wealth_quintile
fourth     3.158234
highest    4.508000
lowest     2.227039
middle     2.738040
second     2.427632
Name: value, dtype: float64

In [23]:
df_non_preg_anemia = pd.DataFrame(non_preg_anemia_prev_averted).rename(columns={'value':'anemia_prev_averted'})
df_non_preg_anemia

,anemia_prev_averted
wealth_quintile,
fourth,0.012215
highest,0.016309
lowest,0.010157
middle,0.011229
second,0.010200


In [24]:
df_non_preg_anemia['pct_anemia_prev_averted'] = non_preg_pct_anemia_prev_averted
df_non_preg_anemia

,anemia_prev_averted,pct_anemia_prev_averted
wealth_quintile,,
fourth,0.012215,3.158234
highest,0.016309,4.508000
lowest,0.010157,2.227039
middle,0.011229,2.738040
second,0.010200,2.427632


In [25]:
df_non_preg_anemia = df_non_preg_anemia.assign(pregnant="not_pregnant").set_index("pregnant", append=True)

In [26]:
df_anemia = pd.concat([df_preg_anemia, df_non_preg_anemia]).sort_index()
df_anemia

anemia_prev_averted  pct_anemia_prev_averted
wealth_quintile pregnant                                                  
fourth          not_pregnant             0.012215                 3.158234
                pregnant                 0.010392                 2.184321
highest         not_pregnant             0.016309                 4.508000
                pregnant                 0.010941                 2.762788
lowest          not_pregnant             0.010157                 2.227039
                pregnant                 0.008547                 1.491027
middle          not_pregnant             0.011229                 2.738040
                pregnant                 0.008620                 1.670973
second          not_pregnant             0.010200                 2.427632
                pregnant                 0.008413                 1.530432